## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [14]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate



#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [15]:
# system: 역할, 상황, 제한 조건, 예시 -> 시스템 메세지
# user: 사용자의 질의 : HumanMessage(..)
# asistant: ai의 답변 : AIMessage(..)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요"),
    ("user", "{user_question}")
])



템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [16]:
sample_prompt = prompt_template.invoke({
    "user_question": "프로그래밍에서 '변수'가 무엇인가요?"
})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'가 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [17]:
model = ChatOllama(
    model="mistral",
    base_url="http://localhost:11434" # ollama 서버 주소
)

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral', base_url='http://localhost:11434')

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

In [18]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 변수는 우리 일상에서 집의 키와 같습니다. 집의 키는 집을 잠긴 문을 열고 들어가는 방법이 될 때 필요합니다. 마찬가지로, 프로그래밍 시 변수는 프로그램이 계산을 수행하거나 데이터를 처리할 때 필요한 데이터를 저장하는 방법입니다. 변수는 특정 데이터를 저장할 수 있는 상자 같은 역할을 합니다. 예를 들어, 사용자가 입력한 이름 또는 수학 문제의 답을 저장할 수 있습니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T06:09:29.5581986Z', 'done': True, 'done_reason': 'stop', 'total_duration': 37349879500, 'load_duration': 18368515700, 'prompt_eval_count': 105, 'prompt_eval_duration': 14307673000, 'eval_count': 218, 'eval_duration': 4666971000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084c8-89fc-72c2-a0de-1b764076e07c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 105, 'output_tokens': 218, 'total_tokens': 323})

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [19]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [20]:
text = parser.invoke(response)

text

' 변수는 우리 일상에서 집의 키와 같습니다. 집의 키는 집을 잠긴 문을 열고 들어가는 방법이 될 때 필요합니다. 마찬가지로, 프로그래밍 시 변수는 프로그램이 계산을 수행하거나 데이터를 처리할 때 필요한 데이터를 저장하는 방법입니다. 변수는 특정 데이터를 저장할 수 있는 상자 같은 역할을 합니다. 예를 들어, 사용자가 입력한 이름 또는 수학 문제의 답을 저장할 수 있습니다.'

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [21]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [ ]:
chain.invoke({
    "user_question": "클래스(Class)에 대해서 알기 쉬운 비유를 통해 설명하세요"
})

' 클래스(Class)는 지금까지 본 여러 개의 동물을 모아 놓은 철 Container 같습니다. 그 안에는 동일한 속성(attribute)과 행동(method)을 가지는 동물들의  blueprint(설계도)가 작성되어 있습니다. 예를 들어, Animal class에는 공통적인 속성들(속성: name, age)과 행동(행동: eat, sleep)이 정의되어 있습니다. 이를 이용하여 새로운 동물을 만들 수 있습니다. 이 과정은 같은 설계도(blueprint)로 만들기 때문에 시간과 노력을 많이 절약할 수 있습니다.'

: 

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)